# Business Sales Performance Analytics
### Future Interns — Data Science & Analytics Internship | Task 1 (FUTURE_DS_01)

**Objective:** Analyze business sales data to identify revenue trends, top-selling products, high-value categories, and regional performance, and translate findings into actionable business recommendations.

**Author:** Future Interns Data Science & Analytics Intern
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly

---
## Table of Contents
1. [Setup & Data Loading](#setup)
2. [Data Cleaning & Preprocessing](#cleaning)
3. [Feature Engineering](#features)
4. [Exploratory Data Analysis](#eda)
5. [Business KPIs](#kpis)
6. [Business Insights](#insights)
7. [Recommendations](#recommendations)
8. [Conclusion](#conclusion)


<a id='setup'></a>
## 1. Setup & Data Loading

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_cleaning import (
    load_raw_data, handle_missing_values, remove_duplicates,
    convert_data_types, detect_outliers, engineer_features, clean_pipeline
)
from analysis import (
    calculate_kpis, region_summary, category_summary, top_products,
    top_customers, monthly_trend, yearly_growth, segment_summary, state_summary
)
import visualization as viz

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RAW_PATH = "../data/sales_data.csv"
df_raw = load_raw_data(RAW_PATH)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

Raw dataset shape: (12120, 21)


,Order ID,Order Date,Ship Date,Customer ID,Customer Name,Segment,Country,State,City,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Mode,Payment Method,Sales Representative
0,ORD-109105,2025-08-05,2025-08-11,CU-10278,Mark Wilson,Consumer,United States,California,San Diego,West,PR-0020,Furniture,Chairs,MeshBack Office Chair,"1,121.49",1,0.05,295.41,Standard Class,Credit Card,Nina Petrova
1,ORD-108870,2023-10-13,2023-10-19,CU-10615,Andrew Lopez,Corporate,United States,California,San Francisco,West,PR-0039,Office Supplies,Binders,Report Cover Binder,52.97,2,0.15,15.55,Second Class,Debit Card,Nina Petrova
2,ORD-100135,2022-08-19,2022-08-25,CU-11694,Steven Harris,Consumer,United States,Oregon,Eugene,West,PR-0039,Office Supplies,Binders,Report Cover Binder,31.06,3,0.15,14.01,Standard Class,PayPal,Maria Silva
3,ORD-110426,2024-05-22,2024-05-28,CU-10447,Joshua Jackson,Corporate,United States,New York,Albany,East,PR-0009,Technology,Computers,UltraSlim Laptop,"5,935.86",6,0.00,"1,768.11",Standard Class,Credit Card,Ethan Wu
4,ORD-108644,2022-08-04,2022-08-09,CU-10316,Deborah King,Corporate,United States,Pennsylvania,Philadelphia,East,PR-0035,Office Supplies,Paper,Recycled Notebook Pack,9.60,2,0.40,-0.48,Standard Class,PayPal,Liam O'Connor


In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 12120 entries, 0 to 12119
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Order ID              12120 non-null  str    
 1   Order Date            12120 non-null  str    
 2   Ship Date             12100 non-null  str    
 3   Customer ID           12120 non-null  str    
 4   Customer Name         12079 non-null  str    
 5   Segment               12120 non-null  str    
 6   Country               12120 non-null  str    
 7   State                 12120 non-null  str    
 8   City                  12120 non-null  str    
 9   Region                12120 non-null  str    
 10  Product ID            12120 non-null  str    
 11  Category              12120 non-null  str    
 12  Sub-Category          12120 non-null  str    
 13  Product Name          12120 non-null  str    
 14  Sales                 12040 non-null  float64
 15  Quantity              12120 no

**Observation:** The raw dataset contains order-level transaction records with customer, product, geographic, and financial fields. Before analysis, we need to check for missing values, duplicates, incorrect data types, and outliers.

In [3]:
print("Missing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
print(f"\nDuplicate rows (Order ID + Product ID): {df_raw.duplicated(subset=['Order ID']).sum()}")

Missing values per column:
Ship Date        20
Customer Name    41
Sales            80
Discount         60
dtype: int64

Duplicate rows (Order ID + Product ID): 120


<a id='cleaning'></a>
## 2. Data Cleaning & Preprocessing

We apply a structured cleaning pipeline: missing value handling, duplicate removal, data type conversion, and outlier detection/treatment.

In [4]:
df = handle_missing_values(df_raw)
df = remove_duplicates(df)
df = convert_data_types(df)
df = detect_outliers(df, column="Sales")
print(f"\nCleaned dataset shape: {df.shape}")

Removed 120 duplicate rows.
Detected 805 outliers in 'Sales' (IQR method).

Cleaned dataset shape: (11980, 22)


<a id='features'></a>
## 3. Feature Engineering

We derive time-based fields (Year, Month, Quarter), shipping performance, Profit Margin, and Average Order Value to support deeper analysis.

In [ ]:
df = engineer_features(df)
df[["Order Date", "Order Year", "Order Month Name", "Order Year-Month",
    "Shipping Days", "Profit Margin (%)", "Average Order Value", "Is Profitable"]].head()

In [ ]:
# Persist the cleaned & engineered dataset for reuse by the dashboard/report scripts
df.to_csv("../data/sales_data_cleaned.csv", index=False)
print("Cleaned dataset saved to ../data/sales_data_cleaned.csv")

<a id='eda'></a>
## 4. Exploratory Data Analysis

The charts below are generated with the reusable `visualization.py` module (Matplotlib/Seaborn) and saved to `../images/` for use in the README and PDF report. Equivalent interactive versions of the key charts are also available in `../dashboard/interactive_dashboard.html` (built with Plotly).

### 4.1 Monthly Revenue & Profit Trend

In [ ]:
viz.plot_monthly_revenue_trend(df, out_dir="../images")
viz.plot_monthly_profit_trend(df, out_dir="../images")

**Insight:** Revenue shows clear month-to-month fluctuation with recurring peaks, typical of B2B/B2C mixed retail seasonality (e.g., back-to-school and year-end buying). Profit closely tracks revenue but dips below zero in months with a high concentration of deep-discount orders, indicating discounting policy is the primary driver of unprofitable months rather than cost inflation.

### 4.2 Sales & Profit by Category

In [ ]:
viz.plot_sales_by_category(df, out_dir="../images")
viz.plot_profit_by_category(df, out_dir="../images")

**Insight:** Technology dominates total revenue and profit contribution, consistent with higher average unit prices. Office Supplies generates the lowest revenue per order but typically carries thinner margins per unit — its contribution depends on order volume rather than order value.

### 4.3 Sales & Profit by Region

In [ ]:
viz.plot_sales_by_region(df, out_dir="../images")
viz.plot_profit_by_region(df, out_dir="../images")

**Insight:** The East region leads in both revenue and profit, while the South region lags behind the other three regions on both metrics. This gap signals an opportunity for a targeted regional sales or marketing push in the South.

### 4.4 Top Products & Top Customers

In [ ]:
viz.plot_top_products(df, out_dir="../images", n=10)
viz.plot_top_customers(df, out_dir="../images", n=10)

**Insight:** A small set of high-ticket Technology products (copiers, workstations, laptops) account for a disproportionate share of revenue — a classic Pareto (80/20) pattern. The top customers list is spread across multiple segments, suggesting revenue is not overly concentrated in a single account (a healthy sign for revenue risk).

### 4.5 Discount vs Profit

In [ ]:
viz.plot_discount_vs_profit(df, out_dir="../images")

**Insight:** Profit declines sharply as discount rates increase past ~20%, with many high-discount orders becoming loss-making (points below the zero line). This is one of the clearest and most actionable patterns in the dataset — discount thresholds should be capped or tied to margin-protection rules.

### 4.6 Correlation Heatmap

In [ ]:
viz.plot_correlation_heatmap(df, out_dir="../images")

**Insight:** Discount shows a negative correlation with Profit and Profit Margin, confirming the pattern observed above. Sales and Profit are positively correlated as expected, while Shipping Days shows negligible correlation with financial outcomes, indicating logistics speed is not currently a profit driver or drag.

### 4.7 Sales & Profit Distribution

In [ ]:
viz.plot_sales_distribution(df, out_dir="../images")
viz.plot_profit_distribution(df, out_dir="../images")

**Insight:** Sales values are right-skewed, dominated by a large volume of small-to-mid orders with a long tail of high-value Technology purchases. The profit distribution is centered near a modest positive value but has a visible left tail of loss-making orders, again tied back to excessive discounting.

### 4.8 Quantity Analysis

In [ ]:
viz.plot_quantity_analysis(df, out_dir="../images")

**Insight:** Office Supplies moves the highest unit volume despite generating the lowest revenue — consistent with its role as a low-price, high-frequency repeat-purchase category.

### 4.9 State-wise Sales

In [ ]:
viz.plot_state_wise_sales(df, out_dir="../images", n=15)

**Insight:** Sales are concentrated in a handful of large states (e.g., California, New York, Texas, Florida), reflecting population and commercial density. Mid-tier states present potential white space for targeted growth campaigns.

### 4.10 Customer Segment Analysis

In [ ]:
viz.plot_customer_segment_analysis(df, out_dir="../images")

**Insight:** The Consumer segment contributes the largest share of total sales, followed by Corporate and then Home Office. Corporate orders, while fewer, tend to have a higher average order value and may warrant a dedicated account-management approach.

### 4.11 Year-wise Growth

In [ ]:
viz.plot_year_wise_growth(df, out_dir="../images")
yearly_growth(df)

**Insight:** Year-over-year revenue growth is uneven, with at least one year showing a decline. This should be cross-checked against external factors (market conditions, product mix changes, discount policy shifts) to determine the root cause before setting next year's targets.

<a id='kpis'></a>
## 5. Business KPIs

In [ ]:
kpis = calculate_kpis(df)
for k, v in kpis.items():
    print(f"{k:45s}: {v}")

### Summary Tables

In [ ]:
print("Region Summary"); display(region_summary(df))

In [ ]:
print("Category Summary (Top 10 Sub-Categories)"); display(category_summary(df).head(10))

In [ ]:
print("Top 10 Products"); display(top_products(df, n=10))

In [ ]:
print("Top 10 Customers"); display(top_customers(df, n=10))

In [ ]:
print("Customer Segment Summary"); display(segment_summary(df))

In [ ]:
print("Top 15 States"); display(state_summary(df, n=15))

<a id='insights'></a>
## 6. Key Business Insights

1. **Technology drives the business.** It is the top revenue and profit-generating category, powered by a small number of high-ticket products (copiers, workstations, premium laptops).
2. **Discounting above ~20% erodes profitability.** A large share of loss-making orders cluster at high discount levels — this is the single largest controllable lever for improving margin.
3. **Regional imbalance exists.** The South region consistently underperforms the East, West, and Central regions on both revenue and profit.
4. **Revenue is right-skewed.** A small number of large orders contribute disproportionately to total revenue, following an 80/20 (Pareto) pattern.
5. **Office Supplies is a volume category, not a value category.** It moves the most units but generates the least revenue, and likely serves as a repeat-purchase, relationship-building category.
6. **Consumer segment leads in volume; Corporate leads in order value.** Different segments may need different sales motions (self-serve vs. account management).
7. **Shipping speed is not a profit driver.** Correlation with financial outcomes is negligible, suggesting current logistics operations are not the constraint on profitability.
8. **Growth is uneven year-over-year.** At least one year shows a revenue decline, warranting a root-cause investigation (seasonality, competition, pricing, or product mix).
9. **A concentrated set of top customers exists but revenue is not dangerously concentrated** in any single account, which is a healthy sign for revenue stability.
10. **Sales are geographically concentrated** in a handful of large states, indicating both a strength (established markets) and an opportunity (underserved mid-tier states).

<a id='recommendations'></a>
## 7. Actionable Business Recommendations

1. **Cap discounts at 20%** for Technology and Furniture orders unless approved by a manager, since profit turns negative beyond this threshold.
2. **Introduce margin-protected discount tiers** (e.g., 0–10% self-serve, 10–20% approval required, 20%+ escalation only) instead of open-ended discounting.
3. **Launch a regional growth campaign in the South**, mirroring tactics that work in the East (top-performing region), including localized promotions and rep incentives.
4. **Double down on top-selling Technology products** — ensure inventory availability and consider bundling accessories to increase average order value.
5. **Reposition Office Supplies as a loyalty/retention category** — use it to drive repeat purchase frequency and cross-sell into higher-margin categories.
6. **Build a dedicated Corporate account-management motion**, since Corporate has the highest average order value and likely responds well to relationship-based selling.
7. **Investigate the year with declining revenue** to isolate whether it was pricing, product mix, discounting, or external market conditions, and adjust strategy accordingly.
8. **Expand marketing investment in mid-tier states** that show moderate but growing sales, rather than only defending top states.
9. **Set up automated monthly profit-trend monitoring** so months trending toward a loss (driven by discount concentration) can be caught and corrected in real time.
10. **Review pricing for the lowest-margin Sub-Categories** identified in the category summary table, especially where Profit is near zero or negative.
11. **Incentivize the top-performing sales representatives'** playbooks and share best practices across the team.
12. **Introduce a customer loyalty program for high-value Consumer segment customers** to protect and grow the largest revenue segment.
13. **Bundle slow-moving Sub-Categories with top-selling products** to increase their sell-through without heavy discounting.
14. **Use shipping mode data to test faster delivery options** in regions with lower sales, to rule out logistics as a hidden barrier to purchase.
15. **Establish a quarterly business review (QBR) cadence** using this dashboard's KPIs (Revenue, Profit Margin, Regional Performance, Top Products) to track the impact of the above initiatives.

<a id='conclusion'></a>
## 8. Conclusion

This analysis of the business sales dataset shows a fundamentally healthy revenue base led by the Technology category, with the largest controllable opportunity being **discount discipline** — a significant share of unprofitable orders is explained by discounts above 20%. Regional performance also shows a clear gap, with the South region trailing the rest of the business. Addressing these two levers (discount policy and regional investment), alongside the other recommendations above, offers a realistic path to improving both revenue growth and profit margin without requiring new product lines.

**Next steps:** Operationalize the KPIs in this notebook into the live `interactive_dashboard.html` for ongoing monitoring, and revisit this analysis quarterly to track the impact of implemented recommendations.

---
*Future Interns — Data Science & Analytics Internship | FUTURE_DS_01*